In [1]:
from langgraph.graph import StateGraph, END

In [7]:
def classification(state):
    text = state.get("user_input", "")

    positive_words = ["재밌", "감동", "좋았", "최고"]
    negative_words = ["지루", "별로", "실망", "최악"]

    # 긍정 단어 포함 여부
    for word in positive_words:
        if word in text:
            return {**state, "label": "positive"}

    # 부정 단어 포함 여부
    for word in negative_words:
        if word in text:
            return {**state, "label": "negative"}

    # 중립
    return {**state, "label": "neutral"}

def positive_answer(state):
    return {**state, "response": "긍정적인 감상평입니다."}

def negative_answer(state):
    return {**state, "response": "부정적인 감상평입니다."}

def neutral_answer(state):
    return {**state, "response": "중립적인 감상평입니다."}

def route_by_label(state):
    return state.get("label", "neutral")

In [8]:
graph = StateGraph(dict)

graph.add_node("classification", classification)
graph.add_node("positive", positive_answer)
graph.add_node("negative", negative_answer)
graph.add_node("neutral", neutral_answer)

graph.set_entry_point("classification")

graph.add_conditional_edges(
    "classification",
    route_by_label,
    {
        "positive": "positive",
        "negative": "negative",
        "neutral":  "neutral"
    }
)

graph.add_edge("positive", END)
graph.add_edge("negative", END)
graph.add_edge("neutral", END)

app = graph.compile()

In [9]:
review = "영화 너무 재밌었어요."

result = app.invoke({"user_input": review})
print("응답:", result.get("response", ""))

응답: 긍정적인 감상평입니다.


In [10]:
review = "영화 너무 지루해요."

result = app.invoke({"user_input": review})
print("응답:", result.get("response", ""))

응답: 부정적인 감상평입니다.


In [11]:
review = "평범해요"

result = app.invoke({"user_input": review})
print("응답:", result.get("response", ""))

응답: 중립적인 감상평입니다.
